# How BLEU is calculated

Using real model predictions from `test_predictions.json`.

In [1]:
import json
from pathlib import Path

path = Path(r"D:\J\Desktop\language_technology\course\projects_AI\mt_oil_no\outputs\exp1_data_scaling\size_13935\seed_42\test_predictions.json")

with open(path, encoding="utf-8") as f:
    data = json.load(f)

sources     = [d["source"]     for d in data]
references  = [d["reference"]  for d in data]
predictions = [d["prediction"] for d in data]

print(f"{len(data)} samples")
print()
print("source    :", sources[0])
print("reference :", references[0])
print("prediction:", predictions[0])

1742 samples

source    : Pressure gradients were not established and the forecasted Permian-Triassic conglomerates were not encountered.
reference : Trykkgradienter ble ikke etablert og de prognoserte konglomeratene av perm-trias alder ble ikke påtruffet.
prediction: Trykkgradienter ble ikke etablert og de prognoserte perm-trias konglomeratene ble ikke påtruffet.


## Step 1 — what is an n-gram?

BLEU works by counting word sequences (n-grams) that appear in both prediction and reference.

In [2]:
pred = predictions[0].split()
ref  = references[0].split()

print("prediction words:", pred)
print()
print("reference words :", ref)

prediction words: ['Trykkgradienter', 'ble', 'ikke', 'etablert', 'og', 'de', 'prognoserte', 'perm-trias', 'konglomeratene', 'ble', 'ikke', 'påtruffet.']

reference words : ['Trykkgradienter', 'ble', 'ikke', 'etablert', 'og', 'de', 'prognoserte', 'konglomeratene', 'av', 'perm-trias', 'alder', 'ble', 'ikke', 'påtruffet.']


In [3]:
from collections import Counter

def ngrams(words, n):
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]

pred_1grams = ngrams(pred, 1)
ref_1grams  = ngrams(ref,  1)

print("prediction unigrams:", pred_1grams)
print()
print("reference unigrams :", ref_1grams)

prediction unigrams: [('Trykkgradienter',), ('ble',), ('ikke',), ('etablert',), ('og',), ('de',), ('prognoserte',), ('perm-trias',), ('konglomeratene',), ('ble',), ('ikke',), ('påtruffet.',)]

reference unigrams : [('Trykkgradienter',), ('ble',), ('ikke',), ('etablert',), ('og',), ('de',), ('prognoserte',), ('konglomeratene',), ('av',), ('perm-trias',), ('alder',), ('ble',), ('ikke',), ('påtruffet.',)]


## Step 2 — count matches

How many prediction n-grams also appear in the reference?

In [4]:
pred_counts = Counter(pred_1grams)
ref_counts  = Counter(ref_1grams)

# clipped count: can't claim more matches than reference has
matches = {gram: min(count, ref_counts[gram]) for gram, count in pred_counts.items()}

print("unigram matches:")
for gram, count in matches.items():
    in_ref = ref_counts[gram]
    print(f"  {gram[0]:30s}  pred={pred_counts[gram]}  ref={in_ref}  matched={count}")

unigram matches:
  Trykkgradienter                 pred=1  ref=1  matched=1
  ble                             pred=2  ref=2  matched=2
  ikke                            pred=2  ref=2  matched=2
  etablert                        pred=1  ref=1  matched=1
  og                              pred=1  ref=1  matched=1
  de                              pred=1  ref=1  matched=1
  prognoserte                     pred=1  ref=1  matched=1
  perm-trias                      pred=1  ref=1  matched=1
  konglomeratene                  pred=1  ref=1  matched=1
  påtruffet.                      pred=1  ref=1  matched=1


In [5]:
total_matches = sum(matches.values())
total_pred    = len(pred_1grams)

precision_1 = total_matches / total_pred

print(f"matched {total_matches} out of {total_pred} unigrams")
print(f"unigram precision = {total_matches}/{total_pred} = {precision_1:.4f}")

matched 12 out of 12 unigrams
unigram precision = 12/12 = 1.0000


## Step 3 — do the same for 2, 3, 4-grams

In [6]:
for n in [1, 2, 3, 4]:
    p_grams = ngrams(pred, n)
    r_grams = ngrams(ref,  n)
    p_counts = Counter(p_grams)
    r_counts = Counter(r_grams)
    matched = sum(min(c, r_counts[g]) for g, c in p_counts.items())
    prec = matched / len(p_grams) if p_grams else 0
    print(f"{n}-gram:  {matched}/{len(p_grams)} matched  →  precision = {prec:.4f}")

1-gram:  12/12 matched  →  precision = 1.0000
2-gram:  8/11 matched  →  precision = 0.7273
3-gram:  6/10 matched  →  precision = 0.6000
4-gram:  4/9 matched  →  precision = 0.4444


## Step 4 — brevity penalty

A short prediction can get high precision by only generating words it's sure about.
BLEU penalises predictions shorter than the reference.

In [7]:
import math

c = len(pred)   # prediction length
r = len(ref)    # reference length

bp = 1.0 if c >= r else math.exp(1 - r/c)

print(f"prediction length = {c}")
print(f"reference length  = {r}")
print(f"brevity penalty   = {bp:.4f}   (1.0 means no penalty)")

prediction length = 12
reference length  = 14
brevity penalty   = 0.8465   (1.0 means no penalty)


## Step 5 — final BLEU score

BLEU = brevity_penalty × geometric_mean(1-gram, 2-gram, 3-gram, 4-gram precision)

In [8]:
precisions = []
for n in [1, 2, 3, 4]:
    p_grams  = ngrams(pred, n)
    r_grams  = ngrams(ref,  n)
    p_counts = Counter(p_grams)
    r_counts = Counter(r_grams)
    matched  = sum(min(c, r_counts[g]) for g, c in p_counts.items())
    precisions.append(matched / len(p_grams) if p_grams else 0)

log_avg   = sum(math.log(p) for p in precisions if p > 0) / 4
bleu_manual = bp * math.exp(log_avg)

print("precisions    :", [round(p, 4) for p in precisions])
print("brevity penalty:", round(bp, 4))
print("BLEU (manual) :", round(bleu_manual, 4))

precisions    : [1.0, 0.7273, 0.6, 0.4444]
brevity penalty: 0.8465
BLEU (manual) : 0.5617


## Step 6 — verify against the library

The library computes corpus-level BLEU across all sentences, not per-sentence.
That's why the number will differ from the single-sentence calculation above.

In [9]:
import evaluate

bleu = evaluate.load("bleu")
result = bleu.compute(
    predictions=predictions,
    references=[[ref] for ref in references],
)

print("corpus BLEU  :", round(result["bleu"], 4))
print("1-gram prec  :", round(result["precisions"][0], 4))
print("2-gram prec  :", round(result["precisions"][1], 4))
print("3-gram prec  :", round(result["precisions"][2], 4))
print("4-gram prec  :", round(result["precisions"][3], 4))
print("brevity pen  :", round(result["brevity_penalty"], 4))
print("length ratio :", round(result["length_ratio"], 4))

d:\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


corpus BLEU  : 0.6187
1-gram prec  : 0.7983
2-gram prec  : 0.6568
3-gram prec  : 0.5651
4-gram prec  : 0.4944
brevity pen  : 1.0
length ratio : 1.0066


## Step 7 — look at a bad prediction vs a good one

In [ ]:
# per-sentence BLEU to find easy vs hard examples
per_sentence = []
for pred, ref in zip(predictions, references):
    r = bleu.compute(predictions=[pred], references=[[ref]])
    per_sentence.append(r["bleu"])

best_idx  = max(range(len(per_sentence)), key=lambda i: per_sentence[i])
worst_idx = min(range(len(per_sentence)), key=lambda i: per_sentence[i])

print("BEST prediction  (BLEU={:.4f}):".format(per_sentence[best_idx]))
print("  ref :", references[best_idx])
print("  pred:", predictions[best_idx])
print()
print("WORST prediction (BLEU={:.4f}):".format(per_sentence[worst_idx]))
print("  ref :", references[worst_idx])
print("  pred:", predictions[worst_idx])